# 📝 과제 01: 루나 팬 대화 챗봇의 평가 기록과 인용 검증

## 만들 결과

루미너스 엔터테인먼트(가상) 운영팀이 버추얼 아이돌 **루나의 팬 대화 챗봇**을 출시하기 전에, 답변이 맞는지, 붙인 인용이 공식 안내 원문과 맞는지, 말투가 루나 챗봇답게 한결같은지를 자동으로 평가해 MLflow에 남기려 합니다. 챗봇은 루나 공식 안내 문서 14편(팬미팅, 굿즈, 팬클럽, 방송 등)을 검색해 답하고, 답변을 부분(segment)으로 나눠 안내에서 가져온 사실을 쓴 부분마다 근거 ID(`docs.jsonl`의 `doc_id`)와 원문 구절을 인용합니다.

- 사람이 쓴 골든셋(정답셋) 12건을 평가 입력 형식으로 바꿉니다.
- 인용 검사 함수와 인용을 붙이는 답변 함수, 인용 검증 코드 채점 함수를 만듭니다.
- 답변 말투를 보는 페르소나 일관성 판사 채점 함수를 만듭니다.
- `mlflow.genai.evaluate`로 평가해 run에 남기고, 통과하지 못한 사례를 이유와 함께 모읍니다.
- 평가 run에 실패 목록 파일과 검토 결과를 덧붙여, MLflow 화면에서도 무엇이 문제였는지 보이게 합니다.

교안 01의 골든셋, 평가 입력 형식, 채점 함수, 인용 검증을 루나 팬 대화 챗봇에 적용합니다. 루나와 소속사, 일정, 금액은 모두 수업용 가상 설정입니다(지난 단원의 루나 설정을 잇습니다).

## 준비: 데이터

- **자료:** `data/과제/`의 공식 안내 문서 `docs.jsonl`(14편), 골든셋 `golden.jsonl`(12건), 미리 만든 문서 벡터 `chroma/`. 필드는 [자료 설명](data/과제/README.md)에 있습니다.
- **순서:** 평가 입력 -> 인용 검사 함수 -> 인용 답변 함수 -> 코드 채점 함수 -> 페르소나 채점 함수 -> 평가와 기록 -> 실패 사례 모으기 -> 검토 결과 남기기
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **판정:** 판사의 판정은 실행마다 조금씩 달라질 수 있습니다. 자가채점은 지표 값이 아니라 결과의 형식(키, 개수, 순서)을 확인합니다.

In [ ]:
import os
import warnings
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# DeepEval 이 사용 기록을 외부로 보내지 않게 합니다. deepeval 을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
# 채점 중 나오는 ipywidgets 설치 권고 경고를 숨깁니다.
warnings.filterwarnings("ignore", message='install "ipywidgets"')
from deepeval.models import OpenAIModel

# material_dir 는 data 와 mlflow.db 가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data" / "과제"

load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: 챗봇 답변을 만듭니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 미리 만들어 둔 문서 벡터와 같은 모델·차원으로 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 짧은 질문이라 자동 길이 검사·분할을 끕니다.

# 판사 모델: 답변 모델과 다른 모델로 채점합니다. 판사를 바꾸면 기준선도 다시 만들어야 합니다.
# gpt-5.6-luna 는 temperature 를 지정할 수 없는 모델이라 지정하지 않습니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-5.6-luna")
judge_llm = OpenAIModel(model=eval_model_name)

# MLflow 자동 기록(autolog)을 끕니다. 켜져 있으면 호출마다 실행 기록(trace)에 단계가 자동으로 생깁니다.
# - mlflow.langchain.autolog: LangChain 체인·프롬프트·모델 호출을 단계마다 입력, 출력, 걸린 시간, 토큰 수와 함께 기록합니다.
# - mlflow.openai.autolog: OpenAI SDK 호출(DeepEval 판사의 호출 포함)을 보낸 메시지, 모델 이름, 응답, 토큰 수와 함께 기록합니다.
# 꺼 두지 않으면 mlflow.genai.evaluate 가 평가 중에 이 둘을 잠시 켭니다. MLflow 3.14 에서는 그 기록이 늦게 닫히면 평가가 실행 기록을 찾지 못해 오류로 멈출 때가 있습니다.
# 실행 기록은 @mlflow.trace 로 직접 남깁니다.
mlflow.langchain.autolog(disable=True)
mlflow.openai.autolog(disable=True)

mlflow.set_tracking_uri(f"sqlite:///{material_dir / 'mlflow.db'}")
experiment = mlflow.set_experiment("luna-rag-eval")
print("답변 모델:", model_name, "/ 판사 모델:", eval_model_name, "/ 실험:", experiment.name)

In [ ]:
import json

import pandas as pd
from langchain_chroma import Chroma


def read_jsonl(path):
    """jsonl 파일(한 줄에 JSON 하나)을 목록으로 읽습니다."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


luna_docs = read_jsonl(data_dir / "docs.jsonl")
luna_golden = read_jsonl(data_dir / "golden.jsonl")
# 인용을 검사할 때 근거 ID로 문서 본문을 찾는 사전입니다.
luna_texts = {}
for doc in luna_docs:
    luna_texts[doc["doc_id"]] = doc["text"]

vector_store = Chroma(collection_name="luna_guides", embedding_function=embedding_model,
                      persist_directory=str(data_dir / "chroma"))
print("문서:", len(luna_docs), "/ 골든 사례:", len(luna_golden))
display(pd.DataFrame(luna_golden)[["case_id", "input", "expected_output", "context", "relevant_doc_ids", "smoke"]])

In [ ]:
def evidence_id(doc):
    """문서에 부여한 근거 ID를 반환합니다."""
    return doc.metadata["source_id"]


@mlflow.trace(span_type="RETRIEVER")
def search_documents(search_query, top_k=4):
    """질문과 가까운 문서 top_k 개를 Document 목록으로 반환합니다."""
    return vector_store.similarity_search(search_query, k=top_k)


def format_documents(docs):
    """검색한 문서를 [근거 ID] 제목과 본문 형식으로 이어 붙입니다."""
    parts = []
    for doc in docs:
        parts.append(f"[{evidence_id(doc)}] {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n".join(parts)


for doc in search_documents("팬클럽 연회비는 얼마예요?"):
    print(evidence_id(doc), "|", doc.metadata["title"])

## 1. 골든셋을 평가 입력 형식으로 바꾸세요

`mlflow.genai.evaluate`는 데이터 한 행을 `inputs`(답변 함수에 넣을 값)와 `expectations`(정답 정보)로 나눠 읽으므로, 골든 사례를 이 모양으로 바꿉니다. 골든 사례 한 건은 `input`(질문), `expected_output`(정답 답변), `context`(정답 근거 문장 목록)를 담습니다. 자료에 없는 질문이면 정답 답변은 확인할 수 없다는 챗봇의 안내 문장입니다.

**작성:** 함수 `to_records(cases)`와 그 결과 `luna_records`

**구현할 처리**
1. `expectations`에 `case_id`, `expected_output`을 골든 사례의 값 그대로 이 순서로 넣습니다. `context`는 사람이 정답을 검토할 때 보는 값이라 넣지 않습니다.
2. 한 행은 `{"inputs": {"question": 질문}, "expectations": ...}`입니다. 질문은 골든 사례의 `input`이고, 키 `question`은 아래에서 만들 답변 함수의 인자 이름과 같아야 합니다.

**반환값:** `luna_records`(dict 12개 리스트, 골든셋과 같은 순서)

In [ ]:
# [작성] 골든 사례를 evaluate 가 읽는 inputs·expectations 형식으로 바꾸세요.
def to_records(cases):
    """골든 사례를 evaluate 가 읽는 inputs·expectations 형식으로 바꿉니다."""
    records = []
    for case in cases:
        # 1. 사례 ID 와 정답 답변
        expectations = ...

        # 2. inputs 와 expectations 두 키의 dict
        records.append(...)
    return records


luna_records = to_records(luna_golden)

In [ ]:
# [자가채점] 행 수·키와 값이 골든셋과 같은지 확인합니다.
assert isinstance(luna_records, list) and len(luna_records) == len(luna_golden), "골든 사례마다 한 행, 12행을 만드세요. `...`를 채웠는지도 확인하세요."
for record, case in zip(luna_records, luna_golden):
    name = case["case_id"]
    assert list(record) == ["inputs", "expectations"], f"{name}: 한 행의 키는 inputs, expectations 두 개입니다."
    assert record["inputs"] == {"question": case["input"]}, f"{name}: inputs 는 키 question 하나에 골든 사례의 질문(input)을 담습니다."
    expectations = record["expectations"]
    assert list(expectations) == ["case_id", "expected_output"], f"{name}: expectations 의 키는 case_id, expected_output 두 개를 이 순서로 둡니다(context 는 넣지 않습니다)."
    assert expectations["case_id"] == name and expectations["expected_output"] == case["expected_output"], f"{name}: case_id 와 expected_output 은 골든셋 값 그대로입니다."
print("평가 입력 확인 완료:", len(luna_records), "행")

## 2. 인용 검사 함수를 만드세요

이 절의 두 함수는 LLM을 부르지 않고 근거 ID와 인용 구절만 봅니다.

**작성:** 함수 `normalize(text)`, `check_citation(citation, source_texts, searched_ids)`

**구현할 처리**
1. `normalize`: 가운뎃점 모양 `･`, `∙`, `・`, `•`를 모두 `·`로 바꾼 뒤, 줄바꿈·탭·연속 공백을 공백 하나로 맞추고 앞뒤 공백을 없앤 문자열을 돌려줍니다.
2. `check_citation`: 인용 하나(`{"evidence_id": ..., "quote": ...}`)를 아래 순서로 검사해 처음 해당하는 결과 문자열을 돌려줍니다.
    - 근거 ID가 `source_texts`에 없으면 `"없는 근거 ID"`
    - 근거 ID가 `searched_ids`(이번에 검색한 근거 ID 목록)에 없으면 `"검색하지 않은 문서"`
    - `normalize`한 구절이 `normalize`한 그 문서 본문 안에 없으면 `"원문에 없는 문구"`
    - 모두 통과하면 `"확인"`

**반환값:** `normalize`는 문자열, `check_citation`은 네 결과 문자열 가운데 하나

In [ ]:
# [작성] 인용 하나를 근거 ID, 인용 문구 순으로 검사하는 함수를 완성하세요.
def normalize(text):
    """뜻과 상관없는 표기 차이(가운뎃점 모양, 줄바꿈, 연속 공백)를 하나로 맞춥니다."""
    # 1-1. 가운뎃점 네 가지 모양을 "·" 로
    ...

    # 1-2. 줄바꿈·연속 공백을 공백 하나로
    return ...


def check_citation(citation, source_texts, searched_ids):
    """인용 하나를 근거 ID, 인용 문구 순으로 검사해 결과 문자열을 돌려줍니다."""
    cited_id = citation["evidence_id"]
    # 2. 없는 근거 ID -> 검색하지 않은 문서 -> 원문에 없는 문구 -> 확인 순서
    ...

In [ ]:
# [자가채점] 일부러 만든 인용 여섯 개로 규칙과 순서를 확인합니다. 실제 답변은 쓰지 않습니다.
fake_texts = {
    "doc-a": "한 계정당 2매까지,\n예매할 수 있습니다. 양도는 할 수 없습니다.",
    "doc-b": "만 7세 미만은 보호자와 함께 입장합니다.",
}
fake_searched = ["doc-a"]
# (인용, 기대 결과, 틀렸을 때 볼 규칙)
fake_citations = [
    ({"evidence_id": "doc-a", "quote": "한 계정당 2매까지, 예매할 수 있습니다"}, "확인", "줄바꿈 차이는 normalize 로 맞춘 뒤 비교합니다."),
    ({"evidence_id": "doc-a", "quote": "한 계정당   2매까지"}, "확인", "연속 공백도 공백 하나로 맞춥니다."),
    ({"evidence_id": "doc-x", "quote": "2매까지"}, "없는 근거 ID", "source_texts 에 없는 근거 ID 는 '없는 근거 ID' 입니다."),
    ({"evidence_id": "doc-b", "quote": "만 7세 미만 어린이"}, "검색하지 않은 문서", "searched_ids 에 없는 문서는 '검색하지 않은 문서' 입니다."),
    ({"evidence_id": "doc-a", "quote": "한 계정당 4매까지"}, "원문에 없는 문구", "숫자가 바뀐 구절은 '원문에 없는 문구' 입니다."),
    ({"evidence_id": "doc-x", "quote": "원문에 없는 문장"}, "없는 근거 ID", "검사는 근거 ID 부터 합니다. 처음 해당하는 결과를 돌려주세요."),
]
for citation, expected, rule in fake_citations:
    result = check_citation(citation, fake_texts, fake_searched)
    assert result == expected, f"{citation} 의 결과가 {result!r} 입니다. 기대: {expected!r}. {rule}"
assert normalize("예매･입장∙굿즈・방송•팬클럽") == "예매·입장·굿즈·방송·팬클럽", "가운뎃점 네 가지 모양을 모두 '·' 로 바꾸세요."
assert normalize(" 앞뒤\t공백\n\n정리 ") == "앞뒤 공백 정리", "줄바꿈·탭·연속 공백은 공백 하나로 바꾸고 앞뒤 공백은 없애세요(split 후 join)."
print("인용 검사 확인 완료: 가짜 인용", len(fake_citations), "개 통과")

## 3. 인용을 붙이는 답변 함수를 만드세요

평가는 답변 함수가 돌려준 dict를 채점하므로, 인용 검사에 필요한 값(답변 부분과 인용, 이번에 검색한 근거 ID)을 모두 dict에 담아 돌려줍니다. 아래 셀은 인용 답변 형식 `CitedAnswer`와 체인 `cited_chain`을 준비합니다.

모델은 답변을 **답변 부분(segment)** 목록으로 돌려주고, 부분마다 그 부분을 뒷받침하는 인용(0개 이상)이 붙습니다. 인용이 붙은 부분이 **주장**이고, 연결 문장이나 확인할 수 없다는 안내처럼 문서에서 가져온 사실이 아닌 부분은 인용이 비어 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class Citation(BaseModel):
    evidence_id: str = Field(description="근거 ID를 넣습니다. 제공된 ID를 철자 그대로 씁니다")
    quote: str = Field(description="그 근거 원문에서 글자 그대로 복사한 구절을 넣습니다. 고치거나 요약하지 않습니다")


class Segment(BaseModel):
    text: str = Field(description="답변의 한 부분(보통 문장 하나)을 넣습니다. segments 의 text 를 순서대로 이으면 사용자에게 보여 줄 답변이 됩니다")
    citations: list[Citation] = Field(description="이 부분을 뒷받침하는 인용 목록입니다. 문서에서 가져온 사실이 아니면(연결 문장, 확인할 수 없다는 안내 등) 빈 목록입니다")


class CitedAnswer(BaseModel):
    segments: list[Segment] = Field(description="답변을 순서대로 나눈 부분 목록입니다")


cited_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 버추얼 아이돌 루나의 팬 대화 챗봇입니다. 팬에게 따뜻한 해요체로, 루나 공식 안내 발췌로만 답하세요. "
               "답변을 부분(segments)으로 나눠 쓰고, 안내에서 가져온 사실을 쓴 부분마다 근거 ID와, 그 문서에서 글자 그대로 복사한 구절을 인용하세요. "
               "구절을 바꿔 쓰거나 요약하지 마세요. "
               "날짜·시각·금액·횟수와 조건을 빠뜨리지 마세요. "
               "안내에 답이 없으면 '제공된 루나 공식 안내에서 확인할 수 없어요.'라는 부분 하나만 인용 없이 쓰세요."),
    ("human", "질문: {question}\n\n공식 안내 발췌:\n{context}"),
])
cited_chain = cited_prompt | llm.with_structured_output(CitedAnswer)

**작성:** 함수 `answer_with_citations(question)`

**구현할 처리**
1. 함수에 `@mlflow.trace`를 붙여 호출 한 번이 실행 기록 하나가 되게 합니다.
2. `search_documents(question)`으로 문서 4편을 검색하고, `format_documents`로 만든 문맥과 질문을 `cited_chain`에 한 번 넘깁니다.
3. 결과를 `model_dump()`로 dict로 바꿉니다(키 `segments`).
4. `segments`의 `text`를 순서대로 공백 하나로 이어(`" ".join`) 키 `answer`에 담습니다. 사용자에게 보여 줄 답변이고, 정답 판정(`correctness`)과 검색 근거 판정(`faithfulness`)은 이 답변을 읽습니다.
5. 키 `searched_ids`에 검색한 문서의 근거 ID 목록(검색 순서)을 더해 돌려줍니다.

**반환값:** `segments`, `answer`, `searched_ids` 세 키의 dict

In [ ]:
# [작성] 질문 하나에 인용을 붙여 답하고 검색한 근거 ID 를 함께 돌려주는 함수를 완성하세요.
# 1. 실행 기록 하나가 되도록 표시
...
def answer_with_citations(question):
    # 2. 검색하고 체인을 한 번 호출
    docs = ...
    result = ...

    # 3. dict 로 바꾸기
    output = ...

    # 4. 부분을 순서대로 이어 사용자에게 보여 줄 답변을 만듭니다.
    texts = []
    ...
    output["answer"] = ...

    # 5. 검색한 근거 ID 목록을 더해 돌려주기
    output["searched_ids"] = ...
    return output


sample = answer_with_citations("루나 첫 팬미팅 「Lumi Night」은 언제, 어디서 시작해요?")
print("답변:", sample["answer"])
for segment in sample["segments"]:
    print("-", segment["text"])
    for citation in segment["citations"]:
        print("   ", citation["evidence_id"], "|", citation["quote"])

In [ ]:
# [자가채점] 돌려준 dict 의 키, 이어 붙인 답변, 검색 근거 ID 를 확인합니다. 답변 문장은 실행마다 달라지므로 보지 않습니다.
assert isinstance(sample, dict) and list(sample) == ["segments", "answer", "searched_ids"], "segments, answer, searched_ids 세 키를 이 순서로 돌려주세요(model_dump 결과에 answer, searched_ids 를 차례로 더합니다)."
searched = sample["searched_ids"]
assert len(searched) == 4 and len(set(searched)) == 4, "searched_ids 는 search_documents 가 돌려준 문서 4편의 근거 ID 목록입니다."
for doc_id in searched:
    assert doc_id in luna_texts, f"{doc_id} 는 문서의 근거 ID 가 아닙니다. evidence_id(doc) 로 꺼내세요."
assert "notice-fm" in searched, "팬미팅 질문인데 팬미팅 공지(notice-fm)가 검색 결과에 없습니다. 질문을 그대로 검색했는지 확인하세요."

assert isinstance(sample["segments"], list), "segments 는 답변 부분 dict 의 목록입니다(model_dump 결과 그대로)."
segment_texts = []
for segment in sample["segments"]:
    assert set(segment) == {"text", "citations"}, "답변 부분 하나는 text, citations 두 키입니다."
    segment_texts.append(segment["text"])
assert sample["answer"] == " ".join(segment_texts), "answer 는 segments 의 text 를 순서대로 공백 하나로 이은 문자열입니다."
print("답변 함수 확인 완료: 검색", len(searched), "편, 답변 부분", len(sample["segments"]), "개")

## 4. 인용 검증 코드 채점 함수를 만드세요

근거 ID가 있는 문서인지, 인용 구절이 원문에 있는지는 코드로 판정할 수 있으므로, LLM 판사 대신 비용이 들지 않고 늘 같은 결과를 내는 코드 채점 함수로 판정합니다.

**작성:** `@scorer`를 붙인 함수 `citation_valid(outputs)`

**구현할 처리**
1. 답변 부분마다, 그 부분에 붙은 인용마다 `check_citation(인용, luna_texts, outputs["searched_ids"])`가 `"확인"`이 아니면 `"{결과}: {근거 ID}"` 형식(예: `원문에 없는 문구: notice-fm`)으로 문제 목록에 더합니다.
2. 문제가 없으면 `Feedback(value=True, rationale="붙은 인용이 모두 근거 ID·문구 대조를 통과함")`, 있으면 `Feedback(value=False, rationale=문제들을 "; "로 이은 문자열)`을 돌려줍니다.

**반환값:** `Feedback`(값은 `True`/`False`, 이유는 통과할 때도 비어 있지 않은 문자열)

인용이 없는 답변 부분은 검사할 인용이 없으므로 이 함수에서는 문제가 아닙니다. 문서에 없는 내용을 인용 없이 쓴 부분은 검색 근거 판정(`faithfulness`)이, 답할 수 있는 질문에 답을 보류한 경우는 정답 판정(`correctness`)이 잡습니다.

In [ ]:
# [작성] 답변에 붙은 인용이 모두 확인되는지 판정하는 코드 채점 함수를 완성하세요.
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer


@scorer
def citation_valid(outputs):
    problems = []
    # 1. 답변 부분마다, 붙은 인용마다 check_citation 결과가 "확인"인지
    ...

    # 2. 문제가 없으면 통과, 있으면 문제들을 "; " 로 이어 실패
    ...

In [ ]:
# [자가채점] 일부러 만든 답변 출력으로 채점 함수의 판정과 이유를 확인합니다. LLM 은 부르지 않습니다.
real_id = "notice-fm"   # 팬미팅 공지
real_quote = "한 계정당 2매까지 예매할 수 있고"
good = {"text": "한 계정당 2매까지 예매할 수 있어요.", "citations": [{"evidence_id": real_id, "quote": real_quote}]}
changed = {"text": "한 계정당 4매까지 예매할 수 있어요.", "citations": [{"evidence_id": real_id, "quote": "한 계정당 4매까지 예매할 수 있고"}]}
titled = {"text": "한 계정당 2매까지 예매할 수 있어요.", "citations": [{"evidence_id": "팬미팅 공지", "quote": real_quote}]}
other = {"text": "팬클럽 혜택이 있어요.", "citations": [{"evidence_id": "faq-fanclub", "quote": "팬클럽"}]}
# 한 부분에 인용이 둘이고 두 번째만 틀린 부분입니다.
two_quotes = {"text": "한 계정당 2매까지, 공식 앱에서 예매해요.",
              "citations": [{"evidence_id": real_id, "quote": real_quote}, {"evidence_id": real_id, "quote": "공식 홈페이지에서 열린다"}]}
# 인용이 없는 부분: 연결 문장과 확인할 수 없다는 안내
bridge = {"text": "팬미팅 예매 방법을 알려 드릴게요.", "citations": []}
abstain = {"text": "제공된 루나 공식 안내에서 확인할 수 없어요.", "citations": []}
passed = "붙은 인용이 모두 근거 ID·문구 대조를 통과함"
fake_outputs = [
    # (이름, outputs, 기대 판정, 기대 이유)
    ("정상", {"segments": [good], "searched_ids": [real_id]}, True, passed),
    ("인용 없는 부분은 통과", {"segments": [bridge, good], "searched_ids": [real_id]}, True, passed),
    ("확인할 수 없다는 안내", {"segments": [abstain], "searched_ids": [real_id]}, True, passed),
    ("구절 바뀜", {"segments": [changed], "searched_ids": [real_id]}, False, "원문에 없는 문구: notice-fm"),
    ("없는 근거 ID", {"segments": [titled], "searched_ids": [real_id]}, False, "없는 근거 ID: 팬미팅 공지"),
    ("검색하지 않은 문서 인용", {"segments": [other], "searched_ids": [real_id]}, False, "검색하지 않은 문서: faq-fanclub"),
    ("한 부분의 두 번째 인용", {"segments": [two_quotes], "searched_ids": [real_id]}, False, "원문에 없는 문구: notice-fm"),
    ("문제 두 개", {"segments": [changed, bridge, other], "searched_ids": [real_id]}, False, "원문에 없는 문구: notice-fm; 검색하지 않은 문서: faq-fanclub"),
]
for name, outputs, value_expected, rationale_expected in fake_outputs:
    valid = citation_valid(outputs=outputs)
    assert isinstance(valid, Feedback), "Feedback(value=..., rationale=...) 을 돌려주세요."
    assert valid.value is value_expected, f"[{name}] citation_valid 는 {value_expected} 여야 합니다. 이유: {valid.rationale}"
    assert valid.rationale == rationale_expected, f"[{name}] 이유가 {valid.rationale!r} 입니다. 기대: {rationale_expected!r}. 구현할 처리의 문구와 '; ' 이음을 확인하세요."
print("코드 채점 함수 확인 완료: 가짜 출력", len(fake_outputs), "개 통과")

## 5. 페르소나 일관성 채점 함수를 만드세요

정답 판정과 검색 근거 판정은 내용만 보므로, 공지문처럼 딱딱한 말투로 답해도 통과합니다. 팬 대화 챗봇은 질문마다 같은 따뜻한 해요체로 답해야 하므로, 말투가 한결같은지(페르소나 일관성)를 따로 채점합니다.

말투에는 정해진 정답 답변이 없어 코드로 판정할 수 없습니다. 지난 단원의 페르소나 G-Eval처럼 판단 단계를 정해 둔 판사로 채점합니다. 아래 셀은 교안 01의 DeepEval 채점 함수 두 개(정답 판정 `correctness`는 G-Eval, 검색 근거 판정 `faithfulness`는 Faithfulness 지표)와 페르소나 판단 단계 `PERSONA_STEPS`를 준비합니다.

In [ ]:
# Feedback 과 scorer 는 위의 코드 채점 함수 셀에서 이미 불러왔습니다.
from deepeval.metrics import GEval, FaithfulnessMetric
from deepeval.metrics.faithfulness import FaithfulnessTemplate
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 결과표에 나오는 네 채점 함수 이름(평가에 넘기는 순서)입니다.
SCORER_NAMES = ["correctness", "faithfulness", "persona_consistency", "citation_valid"]

# 정답 판정의 판단 단계입니다. 정답 답변(expected_output)과 비교합니다.
CORRECTNESS_STEPS = [
    "expected_output 에 적힌 사실이 actual_output 에 모두 들어 있는지 본다.",
    "숫자, 기간, 대상, 조건이 하나라도 다르거나 빠지면 틀린 것으로 본다.",
    "expected_output 이 자료에서 확인할 수 없다는 안내이면, actual_output 도 확인할 수 없다고 답해야 맞다.",
    "판정 이유(reason)는 한국어로 쓴다.",
]

# 페르소나 일관성의 판단 단계입니다. 정답 답변 없이 질문과 답변의 말투만 봅니다.
PERSONA_STEPS = [
    "actual_output 의 문장 끝맺음이 해요체인지 본다. -습니다·-십시오 같은 격식체·안내문체 위주면 감점한다.",
    "팬에게 따뜻하고 친근하게 말하는지 본다. 공지문을 그대로 옮긴 듯한 딱딱한 말투면 감점한다.",
    "짧은 답, 이모지 유무, 꾸밈없는 해요체 평서문, 확인할 수 없다는 해요체 안내는 감점하지 않는다.",
    "정보가 맞는지와 근거가 있는지는 다른 채점 함수가 보므로 이 판정에 반영하지 않는다.",
    "판정 이유(reason)는 한국어로 쓴다.",
]


# 아래 템플릿과 두 채점 함수는 교안 01과 같은 코드입니다.

class KoreanFaithfulnessTemplate(FaithfulnessTemplate):
    """Faithfulness 판사 프롬프트에서 판정 이유를 쓰는 부분만 바꿔, 이유를 한국어로 받습니다."""

    @staticmethod
    def generate_reason(contradictions, score):
        # 기본 프롬프트 끝에 한국어 지시 한 줄을 붙입니다. 주장을 나누고 판정하는 프롬프트는 그대로라 점수는 같습니다.
        return FaithfulnessTemplate.generate_reason(contradictions=contradictions, score=score) + "\n\nreason 값은 한국어로 쓴다."

@scorer
def correctness(inputs, outputs, expectations):
    # 평가는 여러 질문을 동시에 채점하므로, 채점 함수가 불릴 때마다 GEval 지표를 새로 만들어 점수가 섞이지 않게 합니다.
    metric = GEval(
        name="정답 판정",
        evaluation_steps=CORRECTNESS_STEPS,
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        model=judge_llm,
        threshold=0.7,
        async_mode=False,  # 지표 안에서 비동기로 호출하지 않습니다(동시 채점 중 이벤트 루프 오류를 막습니다).
    )
    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], expected_output=expectations["expected_output"]), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)


@scorer
def faithfulness(inputs, outputs):
    # 이번에 검색해 온 문서의 원문 전체를 근거(retrieval_context)로 줍니다.
    retrieval_context = []
    for doc_id in outputs["searched_ids"]:
        retrieval_context.append(luna_texts[doc_id])

    # penalize_ambiguous_claims=True: 문서에 근거가 없는 주장(borderline)도 실패로 셉니다.
    # 기본값 False 는 문서와 모순되는 주장만 실패로 세서, 문서에 없는 말을 덧붙여도 통과합니다.
    # evaluation_template: 판정 이유를 한국어로 받는 프롬프트를 씁니다.
    metric = FaithfulnessMetric(model=judge_llm, threshold=0.7, async_mode=False, penalize_ambiguous_claims=True,
                                evaluation_template=KoreanFaithfulnessTemplate)
    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], retrieval_context=retrieval_context), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)

**작성:** `@scorer`를 붙인 함수 `persona_consistency(inputs, outputs)`

**구현할 처리**
1. 호출될 때마다 `GEval` 지표를 새로 만듭니다. 이름은 `"페르소나 일관성"`, 판단 단계는 `PERSONA_STEPS`, 판사에게 보일 필드는 질문과 답변(`SingleTurnParams.INPUT`, `SingleTurnParams.ACTUAL_OUTPUT`)입니다. `model=judge_llm`, `threshold=0.7`, `async_mode=False`는 `correctness`와 같습니다.
2. `LLMTestCase(input=질문, actual_output=답변)`로 `measure`(`_show_indicator=False`)한 뒤 `Feedback(value=metric.success, rationale=metric.reason)`을 돌려줍니다.

말투만 보므로 정답 답변과 검색 문서는 넘기지 않고, 함수 인자에도 `expectations`를 두지 않습니다.

**반환값:** `Feedback`(값은 `True`/`False`, 이유는 판사가 쓴 문자열)

In [ ]:
# [작성] 답변 말투가 루나 챗봇다운지 판정하는 채점 함수를 완성하세요.
@scorer
def persona_consistency(inputs, outputs):
    # 1. 호출마다 새 GEval 지표(질문과 답변만 보임)
    metric = ...

    # 2. 채점하고 Feedback 으로 돌려주기
    ...
    return ...

In [ ]:
# [자가채점] 말투만 다른 두 답변을 채점해 판정이 갈리는지 확인합니다(판사 호출 2번).
persona_question = {"question": "팬미팅 표는 몇 장까지 예매할 수 있어요?"}
formal = {"answer": "팬미팅 예매는 한 계정당 2매까지 가능합니다. 예매 전 본인 인증을 완료하십시오. 자세한 사항은 공지를 참고하시기 바랍니다."}
warm = {"answer": "한 계정당 2매까지 예매할 수 있어요! 본인 인증만 미리 해 두면 금방 끝나요. 팬미팅에서 꼭 만나요 😊"}
# 정답 답변(expectations) 없이 불러도 동작해야 합니다.
formal_feedback = persona_consistency(inputs=persona_question, outputs=formal)
warm_feedback = persona_consistency(inputs=persona_question, outputs=warm)
for feedback in [formal_feedback, warm_feedback]:
    assert isinstance(feedback, Feedback), "Feedback(value=metric.success, rationale=metric.reason) 을 돌려주세요."
    assert isinstance(feedback.rationale, str) and feedback.rationale, "rationale 에 판사의 이유(metric.reason)를 넣으세요."
assert formal_feedback.value is False, f"격식체 답변이 통과했습니다. evaluation_steps 에 PERSONA_STEPS 를 넘겼는지 확인하세요. 이유: {formal_feedback.rationale}"
assert warm_feedback.value is True, f"해요체 답변이 실패했습니다. evaluation_params 가 질문과 답변 두 개인지 확인하세요. 이유: {warm_feedback.rationale}"
print("격식체:", formal_feedback.value, "|", formal_feedback.rationale)
print("해요체:", warm_feedback.value, "|", warm_feedback.rationale)

## 6. 평가를 실행해 MLflow에 기록하세요

네 채점 함수를 한 run에 함께 기록해 두면, 사례마다 답이 맞는지, 인용이 맞는지, 말투가 한결같은지를 한 화면에서 비교할 수 있습니다.

**작성:** 평가 결과 `luna_result`

**구현할 처리**
1. `mlflow.genai.evaluate`에 `data=luna_records`, `predict_fn=answer_with_citations`, 그리고 채점 함수 네 개를 `correctness`, `faithfulness`, `persona_consistency`, `citation_valid` 순서로 넘깁니다.
2. 결과의 `metrics`를 지표 이름과 소수 셋째 자리 값으로 한 줄씩 출력합니다.

**반환값:** `luna_result`(평가 결과 객체, `metrics`와 `result_df`를 가짐)

In [ ]:
# [작성] 골든셋 12건을 평가해 run 하나에 기록하세요.
# 1. 평가 실행
luna_result = ...

# 2. 지표 출력
for name, value in luna_result.metrics.items():
    print(name, round(float(value), 3))

In [ ]:
# [자가채점] 사례 수와 채점 함수별 지표·열이 모두 있는지 확인합니다. 지표 값은 실행마다 달라지므로 보지 않습니다.
result_df = luna_result.result_df
assert len(result_df) == len(luna_records), "골든 사례 12건이 모두 평가되어야 합니다. data 에 luna_records 를 넘겼는지 확인하세요."
assert sorted(result_df["case_id/value"]) == sorted(case["case_id"] for case in luna_golden), "결과의 case_id 가 골든셋과 같아야 합니다."
for name in SCORER_NAMES:
    assert f"{name}/mean" in luna_result.metrics, f"지표 {name}/mean 이 없습니다. scorers 에 네 채점 함수를 모두 넣었는지 확인하세요."
    assert f"{name}/value" in result_df.columns, f"결과표에 {name}/value 열이 없습니다."
print("평가 run 확인 완료:", luna_result.run_id)

결과는 단원 폴더의 `mlflow.db`, 실험 `luna-rag-eval`에 남았습니다. 교안 01처럼 MLflow 화면을 켜면 이 run의 사례별 판정 이유와 실행 기록을 볼 수 있습니다.

## 7. 통과하지 못한 사례와 이유를 모으세요

지표(통과 비율)만으로는 어느 사례에서 무엇이 틀렸는지 알 수 없으므로, 채점 함수마다 통과하지 못한 사례를 이유와 함께 한 표로 모읍니다.

**작성:** 함수 `collect_failures(result_df)`, 이 평가의 결과 `failure_rows`

**구현할 처리**
1. `result_df`를 `case_id/value` 순으로 정렬해 한 행씩 읽습니다.
2. 행마다 `SCORER_NAMES`(채점 준비 셀에서 정한 네 이름)의 순서로 `이름/value`를 봅니다. 값이 `True` 또는 `1.0`이면 통과입니다(채점 오류로 값이 비어 있으면 통과가 아닙니다).
3. 통과하지 못한 칸마다 `case_id`, `채점 함수`, `판정`(그 칸의 값), `이유`(`이름/rationale` 값) 네 키의 dict를 이 순서로 목록에 더해 돌려줍니다.
4. `luna_result.result_df`를 넘겨 `failure_rows`를 만듭니다.

**반환값:** dict 리스트(case_id 순, 같은 사례 안에서는 `SCORER_NAMES` 순). 모두 통과했으면 빈 리스트입니다.

In [ ]:
# [작성] 채점 함수마다 통과하지 못한 사례를 이유와 함께 모으는 함수를 완성하세요.
def collect_failures(result_df):
    """통과하지 못한 칸마다 case_id, 채점 함수, 판정, 이유 dict 를 모아 돌려줍니다."""
    rows = []
    # 1. case_id 순으로 정렬한 결과표를 한 행씩
    for _, row in ...:
        for name in SCORER_NAMES:
            value = row[f"{name}/value"]
            # 2. 통과한 칸은 건너뛰기
            ...

            # 3. 통과하지 못한 칸은 네 키의 dict 로
            rows.append(...)
    return rows


# 4. 이 평가의 결과표로 실패 목록 만들기
failure_rows = ...
display(pd.DataFrame(failure_rows))

In [ ]:
# [자가채점] 통과·실패·채점 오류를 섞은 가상 결과표로 함수를 확인합니다. 실제 평가는 실패가 없을 수도 있어 가상 표를 씁니다.
fake_table = pd.DataFrame([
    {"case_id/value": "fan-03",
     "correctness/value": False, "correctness/rationale": "사실 하나 빠짐",
     "faithfulness/value": True, "faithfulness/rationale": "근거 있음",
     "persona_consistency/value": False, "persona_consistency/rationale": "격식체 위주",
     "citation_valid/value": False, "citation_valid/rationale": "원문에 없는 문구: notice-x"},
    {"case_id/value": "fan-01",
     "correctness/value": True, "correctness/rationale": "맞음",
     "faithfulness/value": float("nan"), "faithfulness/rationale": "채점 오류",
     "persona_consistency/value": True, "persona_consistency/rationale": "해요체",
     "citation_valid/value": True, "citation_valid/rationale": "통과"},
    {"case_id/value": "fan-02",
     "correctness/value": True, "correctness/rationale": "맞음",
     "faithfulness/value": True, "faithfulness/rationale": "근거 있음",
     "persona_consistency/value": True, "persona_consistency/rationale": "해요체",
     "citation_valid/value": True, "citation_valid/rationale": "통과"},
])
fake_rows = collect_failures(fake_table)
assert isinstance(fake_rows, list) and len(fake_rows) == 4, f"가상 표에서 통과하지 못한 칸은 4개입니다(빈 값 하나, False 셋). 지금 {len(fake_rows)}개입니다."
got = []
for row in fake_rows:
    assert list(row) == ["case_id", "채점 함수", "판정", "이유"], "행의 키는 case_id, 채점 함수, 판정, 이유 순서입니다."
    got.append((row["case_id"], row["채점 함수"], row["이유"]))
expected = [("fan-01", "faithfulness", "채점 오류"), ("fan-03", "correctness", "사실 하나 빠짐"),
            ("fan-03", "persona_consistency", "격식체 위주"), ("fan-03", "citation_valid", "원문에 없는 문구: notice-x")]
assert got == expected, f"{got} 가 기대와 다릅니다. case_id 순, 같은 사례 안에서는 SCORER_NAMES 순이고, 이유는 이름/rationale 값입니다."
for row in fake_rows[1:]:
    assert row["판정"] is False, "판정에는 그 칸의 값을 그대로 넣으세요."
assert isinstance(failure_rows, list), "failure_rows 는 collect_failures(luna_result.result_df) 의 결과입니다."
print("실패 사례 확인 완료: 가상 표", len(fake_rows), "칸, 이 평가", len(failure_rows), "칸")

## 8. 평가 run에 실패 목록과 검토 결과를 남기세요

실패 목록을 노트북에만 두면 다음 사람이 MLflow 화면에서 그 평가를 열어도 무엇이 문제였는지 알 수 없습니다. 교안 01의 1-3절처럼, 끝난 평가 run에 다시 들어가 실패 목록 파일과 검토 결과를 덧붙입니다.

**작성:** 평가 run(`luna_result.run_id`)에 남긴 산출물 `failures.json`, 지표 `failure_count`, 태그 `review_status`

**구현할 처리**
1. `failure_rows`의 행마다 `case_id`, `채점 함수`, `이유` 세 키만 이 순서로 담은 목록 `failure_summary`를 만듭니다(`판정` 값은 파일에 넣지 않습니다).
2. `with mlflow.start_run(run_id=luna_result.run_id):` 안에서 아래 셋을 남깁니다.
    - `mlflow.log_dict(failure_summary, "failures.json")`
    - 지표 `failure_count`: 통과하지 못한 칸 수(`failure_rows`의 길이)
    - 태그 `review_status`: 통과하지 못한 칸이 있으면 `"검토 필요"`, 없으면 `"이상 없음"`

**반환값:** 없음(기록은 평가 run에 남습니다)

In [ ]:
# [작성] 평가 run 에 실패 목록 파일, 실패 칸 수, 검토 상태를 덧붙이세요.
# 1. 파일에 담을 세 키만 고른 목록
failure_summary = []
for row in failure_rows:
    failure_summary.append(...)

# 2. 끝난 평가 run 에 다시 들어가 산출물·지표·태그를 남기기
with ...:
    ...

In [ ]:
# [자가채점] 평가 run 을 다시 읽어 파일·지표·태그가 실패 목록과 맞는지 확인합니다.
review_run = mlflow.get_run(luna_result.run_id)
assert review_run.data.metrics.get("failure_count") == len(failure_rows), "지표 failure_count 에 통과하지 못한 칸 수를 남기세요(같은 평가 run 에)."
expected_status = "이상 없음"
if len(failure_rows) > 0:
    expected_status = "검토 필요"
assert review_run.data.tags.get("review_status") == expected_status, "태그 review_status 는 실패 칸이 있으면 '검토 필요', 없으면 '이상 없음' 입니다."
saved = mlflow.artifacts.load_dict(f"runs:/{luna_result.run_id}/failures.json")
assert isinstance(saved, list) and len(saved) == len(failure_rows), "failures.json 에는 실패 칸마다 한 행이 있어야 합니다."
for saved_row, row in zip(saved, failure_rows):
    assert list(saved_row) == ["case_id", "채점 함수", "이유"], "파일의 행은 case_id, 채점 함수, 이유 세 키를 이 순서로 담습니다."
    assert saved_row["case_id"] == row["case_id"] and saved_row["채점 함수"] == row["채점 함수"], "failure_rows 순서 그대로 담으세요."
print("검토 결과 확인 완료: 실패", len(saved), "칸 / 상태", review_run.data.tags["review_status"])

MLflow 화면에서 이 평가 run을 열면 태그와 지표가 보이고, **Artifacts** 탭에 `failures.json`이 붙어 있습니다.